In [0]:
from pyspark.sql import functions as F

silver_orders = spark.table("workspace.silver.ticket_orders")
silver_items  = spark.table("workspace.silver.ticket_order_items")
silver_events = spark.table("workspace.silver.events")

fact_ticket_sales = (
    silver_items.alias("i")
    .join(silver_orders.alias("o"),
          F.col("i.order_id") == F.col("o.ticket_order_id"))
    .join(silver_events.alias("e"),
          F.col("o.event_id") == F.col("e.event_id"))
    .select(
        # Chaves de dimensão
        F.col("o.ticket_order_id").alias("order_id"),
        F.col("o.event_id"),
        F.col("e.org_id"),
        F.col("o.user_id"),
        F.date_format("o.ordered_date", "yyyyMMdd").cast("int").alias("date_key"),
        # Métricas
        F.col("i.unit_price").alias("unit_price_cents"),
        (F.col("i.unit_price") / 100.0).alias("unit_price_brl"),
        F.col("o.payment_method"),
        F.col("o.status").alias("order_status"),
        # Dimensões degeneradas
        F.col("e.venue_city"),
        F.col("e.venue_state"),
    )
)

(fact_ticket_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_bi.fact_ticket_sales"))